In [3]:
import re
import pandas as pd
from rdflib import Graph, URIRef, Literal, Namespace, RDF
from rdflib.plugins.sparql import prepareQuery
import warnings
import pprint


In [4]:
dataset_folder = 'YAGO4-LP-main'
dataset_name= 'YAGO3-10'

rules_file= f'rule_mining/{dataset_name}/mined_rules-100'
train = f'{dataset_name}/train.txt'
valid = f'{dataset_name}/valid.txt'
test = f'{dataset_name}/test.txt'

In [6]:
#sort and select top N rules

N=13

def parse_rule(line : str):
    conf, rule = line.replace('<=', '').split('\t')[-2:]
    pattern = re.compile(r'(\w+)\((\w+),(\w+)\)')
    #pattern = re.compile(r'([^>]+)\s*\(([^,]+),([^)]+)\)')
    conf = float(conf)
    matches = pattern.findall(rule)
    return (conf,matches)

rules = []
with open(rules_file) as rf:
    for line in rf.readlines():
        rules.append(parse_rule(line))
rules = sorted(rules, key=lambda x: x[0], reverse=True)

top_N = [r[1] for r in rules[:N]]

In [7]:
# Suppress the specific RDFLib warning about serializaition

import logging
rdflib_logger = logging.getLogger('rdflib')
rdflib_logger.setLevel(logging.ERROR)

#parse the graphs

# with warnings.catch_warnings():
#     warnings.simplefilter("ignore")
#     warnings.filterwarnings('ignore')
#
#
#     g.parse(train, format='n3')
#     g.parse(valid, format='n3')
#     g.parse(test, format='n3')
g=Graph()
for fname in [train, valid, test]:
    with open(fname, 'r') as rf:
        for line in rf.readlines():
            s,p,o = line.strip('\n').split('\t')
            g.add((URIRef(s), URIRef(p), URIRef(o)))
print(len(g))

1089040


In [ ]:
#some stats
traindf = pd.read_csv(train, names =['s','p','o'], sep='\t')
traindf['p'].value_counts().reset_index()



In [13]:
def create_query(body):
    q = """
    SELECT *
    WHERE {
    """
    for tp in body[1:]:

        q += f'?{tp[1]} <{tp[0]}> ?{tp[2]} . '
    q += 'filter not exists {{ ?{} <{}> ?{} }}'.format(body[0][1], body[0][0], body[0][2])
    q += """}"""
    return q

#materialize the rules
new_triples = []

for rule in top_N:
    print(rule)
    q=create_query(rule)
    #print(q)
    query_result = g.query(q)
    #print(len(query_result))
    for row in query_result:
        # res = ' '.join([v + ':' + row[v] for v in query_result.vars])
        head_tp = rule[0]
        new_triples.append((row[head_tp[1]], head_tp[0], row[head_tp[2]]))

print(len(new_triples))
new_unique_triples = list(set(new_triples))
print(len(new_unique_triples))

[('hasNeighbor', 'X', 'Y'), ('hasNeighbor', 'Y', 'X')]
[('isMarriedTo', 'X', 'Y'), ('isMarriedTo', 'Y', 'X')]
[('hasGender', 'X', 'Y'), ('hasAcademicAdvisor', 'A', 'X'), ('influences', 'B', 'A'), ('hasGender', 'B', 'Y')]
[('hasGender', 'X', 'Y'), ('hasAcademicAdvisor', 'X', 'A'), ('influences', 'B', 'A'), ('hasGender', 'B', 'Y')]
[('hasGender', 'X', 'Y'), ('hasAcademicAdvisor', 'X', 'A'), ('hasGender', 'A', 'Y')]
[('hasGender', 'X', 'Y'), ('hasAcademicAdvisor', 'A', 'X'), ('hasAcademicAdvisor', 'A', 'B'), ('hasGender', 'B', 'Y')]
[('hasGender', 'X', 'Y'), ('isAffiliatedTo', 'X', 'A'), ('hasGender', 'A', 'Y')]
[('hasGender', 'X', 'Y'), ('hasAcademicAdvisor', 'A', 'X'), ('hasAcademicAdvisor', 'B', 'A'), ('hasGender', 'B', 'Y')]
[('hasGender', 'X', 'Y'), ('isAffiliatedTo', 'X', 'A'), ('created', 'B', 'A'), ('hasGender', 'B', 'Y')]
[('hasGender', 'X', 'Y'), ('hasAcademicAdvisor', 'A', 'X'), ('hasGender', 'A', 'Y')]
[('hasGender', 'X', 'Y'), ('hasAcademicAdvisor', 'A', 'X'), ('influences', 

In [14]:
#add new triples to the graph
new_graph= Graph()
extended_g = Graph()
for s, p, o in g:
    extended_g.add((s, p, o))

with open(f'rule_mining/{dataset_name}/new_triples.txt', 'w') as wf:
    for t in  new_unique_triples:
        new_graph.add((t[0], URIRef(t[1]), t[2]))
        wf.write(f'{str(t[0])}\t{t[1]}\t{str(t[2])}\n')
print(len(extended_g))

#also add taxonomy and schema info

# with open(f'{dataset_folder}/{dataset_name}/ent2classes.txt', 'r') as rf:
#     for l in rf.readlines():
#         s,o = l.split('\t')
#         new_graph.add((URIRef(s), RDF.type, URIRef(o.strip('\n'))))
#         g.add((URIRef(s), RDF.type, URIRef(o.strip('\n'))))

extended_g = extended_g + new_graph
print(len(extended_g))

1089040
1106835


hand-crafted validation for functional properties


In [29]:
shapes = Graph().parse('../datasets/YAGO3-10/yagoSchema.ttl')

query4functional = '''
prefix owl: <http://www.w3.org/2002/07/owl#>
select distinct ?prop where {
    ?prop a owl:FunctionalProperty .
}
'''
res4functional = shapes.query(query4functional)

# Print the results
functional_properties = [str(row.prop).split('/')[-1] for row in res4functional]

In [32]:
def test_functionality(property):
    q= f'''
    ASK {{
      ?subject <{property}> ?object1 .
      ?subject <{property}> ?object2 .
      FILTER (?object1 != ?object2)
    }}
    '''
    return q

for prop in functional_properties:
    answer= g.query(test_functionality(prop))
    if answer.askAnswer == True:
        print(prop)

for prop_name in functional_properties:

    answer= extended_g.query(test_functionality(prop_name))
    if answer.askAnswer == True:
        print(prop_name)

hasGender


Full validation - need to fix it

In [13]:
#validate both graphs

from pyshacl import validate


base_conforms, base_results_graph, base_results_text = validate(data_graph=g,
                                                 shacl_graph=f'Yago4/yago-wd-shapes.nt',
                                                 ont_graph=f'Yago4/yago-wd-class.nt',
                                                                max_validation_depth=32)


#
# extended_conforms, extended_results_graph, extended_results_text = validate(data_graph=extended_g,
#                                                                             shacl_graph=f'Yago4/yago-wd-shapes.nt',
#                                                                             ont_graph=f'Yago4/yago-wd-schema.nt')

/Users/thezamp/miniconda3/envs/calibration/lib/python3.10/site-packages/pyshacl/constraints/core/shape_based_constraints.py:113: ShapeRecursionWarning: Warning, A Recursive Shape was detected executing a recursive validation sequence 20 levels deep. Backing out.
<NodeShape http://schema.org/CreativeWork>-><PropertyConstraintComponent on <NodeShape http://schema.org/CreativeWork>>-><PropertyShape http://yago-knowledge.org/value/shape-prop-schema-CreativeWork-schema-copyrightHolder>-><OrConstraintComponent on <PropertyShape http://yago-knowledge.org/value/shape-prop-schema-CreativeWork-schema-copyrightHolder>>-><NodeShape http://yago-knowledge.org/value/sh-node-schema-Organization>-><NodeConstraintComponent on <NodeShape http://yago-knowledge.org/value/sh-node-schema-Organization>>-><NodeShape http://schema.org/Organization>-><PropertyConstraintComponent on <NodeShape http://schema.org/Organization>>-><PropertyShape http://yago-knowledge.org/value/shape-prop-schema-Organization-schema-fo

KeyboardInterrupt: 

In [11]:
query_shacl = """
    PREFIX sh: <http://www.w3.org/ns/shacl#>
    SELECT distinct ?violation WHERE {
        ?s sh:resultMessage ?violation.
    }
"""

qres_base = base_results_graph.query(query_shacl)

# Print the results
for row in qres_base:
    print(f"{row.violation}")

print(len(base_results_graph))


2


'Validation Report\nConforms: True\n'

In [ ]:

new_conforms, new_results_graph, new_results_text = validate(data_graph=new_graph,
                                                                            shacl_graph=f'Yago4.5/yago-schema.ttl', ont_graph=f'Yago4.5/yago-taxonomy.ttl', debug=True)
qres_new = new_results_graph.query(query_shacl)



# Print the results
for row in qres_new:
    print(f"{row.violation}")

print(len(new_results_graph))

In [ ]:
print(len(g))
print(len(extended_g))

In [ ]:
qres_extended = extended_results_graph.query(query_shacl)

# Print the results
for row in qres_extended:
    print(f"{row.violation}")

print(len(extended_results_graph))



In [ ]:
info= """
PREFIX sh: <http://www.w3.org/ns/shacl#>
PREFIX yago: <http://yago-knowledge.org/resource/>
PREFIX schema: <http://schema.org/>

SELECT distinct * WHERE {
    ?child schema:parent ?parent .
    ?parent schema:gender ?g .
    filter not exists {{ ?child schema:gender ?g .}}
}
"""

info_res = g.query(info)
for row in info_res:
    print(row.child + '\t' + row.g)


In [ ]:
list(info_res)

Exploring yago4.5 tiny

In [ ]:
yago45 = Graph()
yago45.parse('Yago4.5/yago-tiny.ttl', format='n3')

In [ ]:
materialize.ipynb